# ML Lab 7

**Aim:** Build ensemble learning models using Bagging, Random Forest, and AdaBoost techniques and evaluate their performance using appropriate metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Feature shape:', X.shape)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

## Experiment 1: Bagging Classifier
1. Train a Bagging classifier using Decision Tree as base estimator.
2. Predict class labels for the test data.
3. Evaluate using Accuracy, Confusion Matrix, Classification Report, Precision-Recall Curve.

In [ ]:
# Bagging Classifier
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50, random_state=42
)
bag_model.fit(X_train, y_train)

y_pred_bag = bag_model.predict(X_test)
y_prob_bag = bag_model.predict_proba(X_test)[:, 1]

acc_bag_train = accuracy_score(y_train, bag_model.predict(X_train))
acc_bag_test = accuracy_score(y_test, y_pred_bag)
cm_bag = confusion_matrix(y_test, y_pred_bag)

print('Bagging Train Accuracy:', round(acc_bag_train, 4))
print('Bagging Test Accuracy :', round(acc_bag_test, 4))
print('\nConfusion Matrix:\n', cm_bag)
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred_bag, target_names=data.target_names))

In [ ]:
# Bagging Precision-Recall Curve
precision_bag, recall_bag, _ = precision_recall_curve(y_test, y_prob_bag)

plt.figure(figsize=(7, 5))
plt.plot(recall_bag, precision_bag, color='navy', label='Bagging')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Bagging)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Experiment 2: Random Forest Classifier
1. Train a Random Forest classifier.
2. Predict class labels for the test data.
3. Evaluate using Accuracy, Confusion Matrix, Classification Report, Feature Importance Visualization.

In [ ]:
# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

acc_rf_train = accuracy_score(y_train, rf_model.predict(X_train))
acc_rf_test = accuracy_score(y_test, y_pred_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print('Random Forest Train Accuracy:', round(acc_rf_train, 4))
print('Random Forest Test Accuracy :', round(acc_rf_test, 4))
print('\nConfusion Matrix:\n', cm_rf)
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred_rf, target_names=data.target_names))

In [ ]:
# Feature Importance Visualization
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.bar(range(15), importances[indices], color='forestgreen')
plt.xticks(range(15), X.columns[indices], rotation=45, ha='right')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## Experiment 3: AdaBoost Classifier
1. Train an AdaBoost classifier using Decision Tree as base estimator.
2. Predict class labels for the test data.
3. Evaluate using Accuracy, Confusion Matrix, Classification Report.

In [ ]:
# AdaBoost Classifier
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=50, random_state=42
)
ada_model.fit(X_train, y_train)

y_pred_ada = ada_model.predict(X_test)
y_prob_ada = ada_model.predict_proba(X_test)[:, 1]

acc_ada_train = accuracy_score(y_train, ada_model.predict(X_train))
acc_ada_test = accuracy_score(y_test, y_pred_ada)
cm_ada = confusion_matrix(y_test, y_pred_ada)

print('AdaBoost Train Accuracy:', round(acc_ada_train, 4))
print('AdaBoost Test Accuracy :', round(acc_ada_test, 4))
print('\nConfusion Matrix:\n', cm_ada)
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred_ada, target_names=data.target_names))

## Experiment 4: Compare Bagging, Random Forest, and AdaBoost
1. Bar chart comparing accuracy of all models.
2. Confusion Matrix Heatmap for each model.
3. Performance comparison analysis.

In [ ]:
# 1) Accuracy bar chart
models = ['Bagging', 'Random Forest', 'AdaBoost']
train_acc = [acc_bag_train, acc_rf_train, acc_ada_train]
test_acc = [acc_bag_test, acc_rf_test, acc_ada_test]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, train_acc, width, label='Training Accuracy')
plt.bar(x + width / 2, test_acc, width, label='Testing Accuracy')
plt.xticks(x, models)
plt.ylim(0.0, 1.05)
plt.ylabel('Accuracy')
plt.title('Ensemble Models: Training vs Testing Accuracy')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# 2) Side-by-side confusion matrix heatmaps
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.heatmap(cm_bag, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Bagging Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

sns.heatmap(cm_ada, annot=True, fmt='d', cmap='Oranges', ax=axes[2])
axes[2].set_title('AdaBoost Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# 3) Performance comparison table
comparison_df = pd.DataFrame({
    'Model': ['Bagging', 'Random Forest', 'AdaBoost'],
    'Train Accuracy': [acc_bag_train, acc_rf_train, acc_ada_train],
    'Test Accuracy': [acc_bag_test, acc_rf_test, acc_ada_test],
})
print(comparison_df)

## Answers
1. **Which ensemble model achieved the highest accuracy?** Random Forest achieved the highest test accuracy among the three ensemble models.